# Experimento — Perceptron multicamadas (MLP)

Rede treinada do zero para prever `Churn`. O estudo avança em **blocos sequenciais**: cada bloco testa uma intervenção sobre o campeão do anterior e só a aceita se superar o ruído, para sabermos o que melhora o modelo e o que não melhora.

1. **Referência e busca:** o MLP padrão, as referências e o Optuna, com KS como critério e as 5 dobras oficiais.
2. **Erros consistentes:** quais clientes várias "lentes" erram juntas e o que isso revela sobre como os modelos veem os dados. Cada padrão vira uma hipótese.
3. **Insights em sequência:** cada hipótese é um bloco, que herda o campeão do anterior.
4. **Registro evolutivo:** o que cada etapa do pré-processamento muda no MLP e no XGBoost.

**Regra de aceite:** ganhar em pelo menos 80% das divisões repetidas do treino e ter ganho médio maior que a régua de ruído (quanto o KS do próprio campeão muda só de trocar a semente).

O teste só é usado no fim, uma vez, com a sequência congelada e `REVELAR_TESTE = True`. Até lá tudo usa apenas o treino.

## Como executar

### No Kaggle (execução principal)

1. **Notebook:** *Code → New Notebook → File → Import Notebook* → `experimentos/mlp/mlp.ipynb`.
2. **Dados:** *Add Input* → dataset `customers-churned-in-telecom-services`. Sem ele, o notebook tenta baixar via `kagglehub`.
3. **Secrets:** *Add-ons → Secrets* → `GITHUB_TOKEN` e `WANDB_API_KEY`, marcados como anexados. Não suba o `.env` para o Kaggle.
4. **Settings:** *Internet* **On**. O MLP é pequeno, então o acelerador é opcional.
5. Rode primeiro com `MODO_TESTE = True`. Depois, com `False`, use *Save Version → Save & Run All (Commit)*.

Para retomar uma execução: use o mesmo `RUN_NAME` e `RESUME_FROM = "github"`.

### Localmente

Preencha o `.env` da raiz do repositório (é opcional) e rode de qualquer pasta do projeto. O CSV é procurado em `dados/telco_ibm.csv`.

# 0. Preparação do ambiente

In [1]:
# ===== Configuração da execução (edite aqui) =====
REPO_URL = "https://github.com/diegoflyra/churn_project.git"
BRANCH = "diego/mlp"             # branch com o código
RESULTS_REPO_URL = REPO_URL      # onde espelhar os resultados ("" = não enviar ao GitHub)
RESULTS_BRANCH = "resultados"    # branch órfã só de resultados
RUN_NAME = ""                    # vazio = "mlp-AAAAMMDD-HHMM"; fixe um nome para retomar
RESUME_FROM = ""                 # "" | "github" | pasta com os resultados de uma execução anterior
CSV_PATH = "/kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv" # caminho do CSV; vazio = procurar no repositório e em /kaggle/input
N_TRIALS = 100                   # mesmo N para todos os modelos (pré-proposta)
N_SEMENTES = 5                   # sementes da avaliação final e das lentes de erro
K_CONFIGS = 5                    # melhores configurações do estudo usadas como lentes de erro
REPETICOES = 5                   # divisões de treino repetidas no registro evolutivo e nas hipóteses
REVELAR_TESTE = False            # True só com a configuração congelada: o teste é usado uma única vez
USAR_WANDB = True                # False = não envia nada ao W&B, mesmo com a chave
USAR_GITHUB = True               # False = não espelha os resultados no GitHub, mesmo com o token
MODO_TESTE = False               # True = passada rápida para validar o notebook; os números não servem como resultado
HORAS_LIMITE = 10.5              # depois disso não começa trials novos e encerra limpo (o Kaggle corta em 12 h)
BACKUP_MIN = 30                  # envia os resultados ao GitHub a cada N minutos durante a busca

In [2]:
import importlib.util, os, shutil, subprocess, sys, time
from pathlib import Path

INICIO = time.time()


def _load_dotenv(path):
    """Tokens em um .env local (GITHUB_TOKEN, WANDB_API_KEY); nunca sobrescreve variáveis já definidas."""
    if os.path.isfile(path):
        for line in open(path, encoding="utf-8"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#") and value.strip() and not os.environ.get(key.strip()):
                os.environ[key.strip()] = value.strip().strip("'").strip('"')


def _secret(name):
    value = os.environ.get(name, "").strip()
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return ""


def _is_repo(p):
    return (p / "comum.py").exists() and (p / "experimentos" / "mlp" / "modelo.py").exists()


local = next((p for p in [Path.cwd(), *Path.cwd().parents] if _is_repo(p)), None)
if local:
    REPO_DIR = local
    _load_dotenv(REPO_DIR / ".env")
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    print(f"Código: cópia local em {REPO_DIR}")
else:
    GITHUB_TOKEN = _secret("GITHUB_TOKEN")
    REPO_DIR = Path("/tmp/churn_project")
    url = REPO_URL.replace("https://", f"https://x-access-token:{GITHUB_TOKEN}@", 1) if GITHUB_TOKEN else REPO_URL
    print("GitHub: usando GITHUB_TOKEN" if GITHUB_TOKEN else "GitHub: clone sem token")
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    if subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)]).returncode != 0:
        raise RuntimeError(f"Falha ao clonar a branch {BRANCH}. Confira se ela foi publicada; repositório privado exige GITHUB_TOKEN.")
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL])  # o token não fica salvo no clone
    print(f"Código: {REPO_DIR}")
os.chdir(REPO_DIR)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

# Instala só o que falta: o Kaggle já traz versões compatíveis com a GPU dele
IMPORTS = {"imbalanced-learn": "imblearn", "scikit-learn": "sklearn"}
faltam = [p.strip() for p in open("requirements.txt", encoding="utf-8")
          if p.strip() and importlib.util.find_spec(IMPORTS.get(p.strip(), p.strip())) is None]
if faltam:
    print("Instalando:", ", ".join(faltam))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltam])

GitHub: usando GITHUB_TOKEN
Código: /tmp/churn_project
64a514b fix: add csv path on mlp experiment


In [3]:
import json

sys.path[:0] = [str(REPO_DIR), str(REPO_DIR / "experimentos" / "mlp")]
import fluxo

WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if os.path.isdir("/content") else str(REPO_DIR)
RUN_NAME = RUN_NAME or time.strftime("mlp-%Y%m%d-%H%M")
if MODO_TESTE and not RUN_NAME.startswith("teste-"):
    RUN_NAME = "teste-" + RUN_NAME
# O teste nunca se mistura com a execução real, e rodar localmente não sobrescreve o que está versionado em resultados/
PASTA = os.path.join(WORKDIR, "execucoes", RUN_NAME) if WORKDIR == str(REPO_DIR) else os.path.join(WORKDIR, "resultados_" + RUN_NAME)
os.makedirs(PASTA, exist_ok=True)
os.environ["CHURN_RESULTADOS"] = PASTA
os.environ["CHURN_CSV"] = fluxo.achar_csv(CSV_PATH)
if MODO_TESTE:
    os.environ["CHURN_MAX_EPOCAS"] = "5"
    N_TRIALS, N_SEMENTES, K_CONFIGS, REPETICOES = 3, 2, 2, 2
    print("MODO_TESTE: 3 trials, 5 épocas. Os números não servem como resultado.")

if not USAR_GITHUB:
    RESULTS_REPO_URL = ""
if RESUME_FROM == "github":
    fluxo.baixar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)
elif RESUME_FROM:
    shutil.copytree(RESUME_FROM, PASTA, dirs_exist_ok=True)


def salvar():
    return fluxo.salvar_github(PASTA, RUN_NAME, RESULTS_REPO_URL, GITHUB_TOKEN, RESULTS_BRANCH)


def salvar_json(nome, obj):
    json.dump(obj, open(os.path.join(PASTA, nome), "w"), indent=1)
    salvar()


def tempo_restante():
    return HORAS_LIMITE * 3600 - (time.time() - INICIO)


def continuar():
    return tempo_restante() > 0


import numpy as np, optuna, pandas as pd, torch
from scipy.stats import ks_2samp
import comum, modelo as M, avaliar as A, erros as ER, evolucao as EV, blocos as BL

pd.set_option("display.max_columns", None, "display.width", 200, "display.max_colwidth", 60)
print(f"RUN_NAME = {RUN_NAME} | resultados em {PASTA}")
print(f"CSV: {os.environ['CHURN_CSV']} | device: {M.DEVICE} | GitHub: {'sim' if GITHUB_TOKEN and RESULTS_REPO_URL else 'não'}")

wandb_run = None
wandb_key = _secret("WANDB_API_KEY") if USAR_WANDB else ""
if wandb_key or os.environ.get("WANDB_MODE") == "offline":
    import wandb
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    wandb_run = wandb.init(project="churn-if702", entity=_secret("WANDB_ENTITY") or None, name=RUN_NAME, dir=WORKDIR,
                           group="mlp", config=dict(modelo="mlp", n_trials=N_TRIALS, lote=M.LOTE, paciencia=M.PACIENCIA, teste=MODO_TESTE))
else:
    print("W&B desativado (sem chave ou USAR_WANDB = False).")


def _apos_bloco(r):
    if wandb_run is not None:
        wandb_run.log({"bloco": r["nome"], "regua": r["regua"], "ks_campeao": max(l["ks"] for l in r["tabela"] if l["decisao"] in ("base", "ACEITA"))})
    salvar()


BL.configurar(PASTA, REPETICOES, _apos_bloco)

RUN_NAME = mlp-20261103-0337 | resultados em /kaggle/working/resultados_mlp-20261103-0337
CSV: /kaggle/input/datasets/diegolyra/churn-ibm/telco_ibm.csv | device: cuda | GitHub: sim


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: dfal (models-universidade-federal-de-pernambuco) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261103_033825-jsokp2e4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run mlp-20261103-0337
wandb: ⭐️ View project at https://wandb.ai/models-universidade-federal-de-pernambuco/churn-if702
wandb: 🚀 View run at https://wandb.ai/models-universidade-federal-de-pernambuco/churn-if702/runs/jsokp2e4


In [4]:
# Verificações rápidas antes de gastar tempo de treino
folds, final, colunas = comum.carregar_dobras()
ind_val, _ = comum.indices_dobras()
df_ = comum.carregar_df()
for i, f in enumerate(folds):
    assert np.array_equal(df_.loc[ind_val[i], "Churn"].values, f[3].astype(int)), "índices fora de ordem"
    assert abs(f[1].mean() - 0.5) < 1e-6, "treino da dobra deveria estar balanceado"
print(f"{len(df_)} clientes | {len(colunas)} variáveis | validação por dobra: {[len(f[2]) for f in folds]} | índices conferidos")
rng = np.random.default_rng(1); y_ = (rng.random(2000) < .265).astype(int); p_ = np.clip(rng.normal(.3 + .25 * y_, .2), 0, 1)
assert abs(comum.ks(y_, p_) - ks_2samp(p_[y_ == 1], p_[y_ == 0]).statistic) < 1e-9
print("KS conferido contra scipy")

7043 clientes | 28 variáveis | validação por dobra: [1057, 1057, 1056, 1056, 1056] | índices conferidos
KS conferido contra scipy


# Bloco 0 — Referências

**Pergunta:** onde partimos? O MLP padrão da pré-proposta (1 camada, 10 unidades), uma regressão logística e um XGBoost nas 5 dobras oficiais.

In [5]:
from sklearn.linear_model import LogisticRegression
ref = {"mlp_padrao": [comum.ks(f[3], M.fit(M.MLP_PADRAO, f, [f[2]], 42 + i)[0][0]) for i, f in enumerate(folds)],
       "regressao_logistica": [comum.ks(f[3], LogisticRegression(max_iter=3000).fit(f[0], f[1]).predict_proba(f[2])[:, 1]) for f in folds],
       "xgboost": [comum.ks(f[3], ER._xgb(42).fit(f[0], f[1]).predict_proba(f[2])[:, 1]) for f in folds]}
display(pd.DataFrame(ref, index=[f"dobra {i}" for i in range(5)]).T.assign(media=lambda d: d.mean(axis=1), dp=lambda d: d.iloc[:, :5].std(axis=1)).round(4))
camp = BL.campeao_inicial(M.MLP_PADRAO)

,dobra 0,dobra 1,dobra 2,dobra 3,dobra 4,media,dp
mlp_padrao,0.5695,0.5628,0.5220,0.5208,0.5156,0.5381,0.0258
regressao_logistica,0.5746,0.5624,0.5169,0.5190,0.5236,0.5393,0.0271
xgboost,0.5429,0.5482,0.4927,0.4983,0.5403,0.5245,0.0267


# Bloco 1 — Busca de hiperparâmetros no Optuna

**Pergunta:** quanto a busca melhora o MLP padrão?

Nove hiperparâmetros: os 8 da pré-proposta (camadas, unidades, ativação, otimizador, taxa, dropout, L2 e perda) mais a **paciência** {5, 10, 20, 30}. Lote 256 e no máximo 10.000 épocas ficam fixos. O trial 0 é a configuração padrão. O estudo fica em SQLite e é retomável.

A busca escolhe nas dobras oficiais, então o melhor trial tende a estar otimista. Por isso, a comparação com o padrão é refeita em divisões novas do treino.

In [6]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
cb = fluxo.callback_trial(wandb_run, salvar, BACKUP_MIN)
st = M.estudo(N_TRIALS, timeout=max(tempo_restante(), 1), callbacks=[cb])
salvar()
ok = [t for t in st.trials if t.state.name == "COMPLETE"]
print(f"{len(ok)} completos, {len(st.trials) - len(ok)} podados ou interrompidos | tempo de parede: {st.user_attrs.get('tempo_parede_s', 0) / 60:.1f} min")
print(f"KS trial 0 (padrão) = {st.trials[0].value:.4f} -> melhor trial = {st.best_value:.4f} (trial {st.best_trial.number})")
print("Melhor configuração:", st.best_params)
df = st.trials_dataframe()
df = df[["number", "state", "value"] + [c for c in df.columns if c.startswith("params_")]]
display(df.sort_values("value", ascending=False).head(10))

GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403

42 completos, 58 podados ou interrompidos | tempo de parede: 27.7 min
KS trial 0 (padrão) = 0.5381 -> melhor trial = 0.5451 (trial 87)
Melhor configuração: {'camadas': 1, 'unidades': 5, 'ativ': 'relu', 'otim': 'adam', 'lr': 0.0004403287790754724, 'dropout': 0.1877796113284086, 'l2': 0.0005634765738856296, 'perda': 'ce', 'paciencia': 10}


,number,state,value,params_ativ,params_camadas,params_dropout,params_l2,params_lr,params_otim,params_paciencia,params_perda,params_unidades
78,78,PRUNED,0.573139,relu,1,0.303545,0.001455,0.000207,adam,10,ce,5
80,80,PRUNED,0.571979,relu,1,0.168362,0.000489,0.000357,adam,10,ce,8
97,97,PRUNED,0.571608,relu,1,0.199319,0.000561,0.000295,adam,10,ce,5
90,90,PRUNED,0.571360,relu,1,0.240228,0.000407,0.000407,adam,10,ce,6
70,70,PRUNED,0.571335,relu,1,0.388059,0.002088,0.000135,adam,10,ce,6
93,93,PRUNED,0.570750,relu,1,0.127390,0.001575,0.000291,adam,10,ce,5
99,99,PRUNED,0.569982,relu,1,0.192865,0.001271,0.000217,adam,10,ce,7
89,89,PRUNED,0.569154,relu,1,0.185276,0.002226,0.000619,adam,10,ce,5
44,44,PRUNED,0.567985,relu,1,0.188268,0.001039,0.000168,adam,10,ce,12
91,91,PRUNED,0.567836,relu,1,0.119159,0.000296,0.000412,adam,10,ce,8


In [7]:
# Melhor KS por valor de cada variável categórica, quantos trials cada valor recebeu e importâncias
linhas = [dict(hiperparametro=h, valor=str(t.params[h]), ks=t.value) for t in ok for h in ("ativ", "otim", "camadas", "perda", "paciencia")]
display(pd.DataFrame(linhas).groupby(["hiperparametro", "valor"]).ks.agg(["count", "max", "median"]).round(4))
try:
    print("Importâncias:", {k: round(v, 3) for k, v in optuna.importance.get_param_importances(st).items()})
except Exception as e:
    print("Importâncias indisponíveis:", e)

count     max  median
hiperparametro valor                           
ativ           logistica      3  0.5353  0.4711
               relu          38  0.5451  0.5414
               tanh           1  0.5361  0.5361
camadas        1             41  0.5451  0.5411
               2              1  0.5361  0.5361
otim           adadelta       3  0.5363  0.5361
               adam          34  0.5451  0.5423
               rmsprop        1  0.5363  0.5363
               sgd            4  0.5353  0.4798
paciencia      10            32  0.5451  0.5425
               20             2  0.5363  0.5037
               30             2  0.5353  0.5119
               5              6  0.5402  0.5378
perda          ce            38  0.5451  0.5414
               mse            4  0.5363  0.5362

Importâncias: {'dropout': 0.248, 'l2': 0.246, 'lr': 0.212, 'unidades': 0.182, 'otim': 0.033, 'ativ': 0.033, 'paciencia': 0.025, 'camadas': 0.018, 'perda': 0.003}


In [8]:
if continuar():
    camp, t = BL.rodar("b01_busca", "O melhor trial supera o MLP padrão em divisões novas?", {"melhor do Optuna": dict(hp=st.best_params)}, camp)
    display(t.round(4))

b01_busca: O melhor trial supera o MLP padrão em divisões novas?
regua de ruido = 0.0028 | campeao mantido | 167s
GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



,variante,ks,delta,dp_delta,ganhou,decisao
0,(campeao atual),0.5400,0.0000,0.0000,,base
1,melhor do Optuna,0.5397,-0.0002,0.0033,1/5,rejeitada


### 📝 Análise — Blocos 0 e 1

- **Quanto a busca ganhou sobre o padrão, em divisões novas?** _…_
- **Quais hiperparâmetros importaram? A paciência importou?** _…_
- **Algum otimizador recebeu poucos trials para ser comparado?** _…_

# Erros consistentes do campeão

Um erro de um modelo pode ser acaso. Um erro que sementes, configurações e famílias diferentes cometem juntas diz algo sobre os dados.

**Lentes:** o campeão em várias sementes, as melhores configurações do estudo, uma regressão logística e um XGBoost, nas mesmas dobras. Cada lente decide no próprio limiar de KS máximo. Só entram previsões de validação.

**Três perguntas, nesta ordem:**

1. **Os erros são compartilhados?** `todas_erram` muito acima de `esperado_se_independentes` indica que sim, e `concordancia` mostra entre quais lentes.
2. **Quais variáveis os modelos usam?** `perfil` compara quem é sempre errado com quem é sempre acertado na mesma classe. Esses grupos estão em lados opostos da fronteira, então as diferenças são grandes por construção.
3. **Existe informação para separar os erros?** `mesmo_lado` compara clientes que os modelos veem do mesmo jeito, com rótulos opostos. Um segmento com `q_bh` baixo aqui é a melhor fonte de hipóteses. Se nada aparecer, o limite pode estar na base.

In [9]:
def analisar_erros(c, rotulo):
    fs, _ = BL.dobras_de(c)
    outras = [p for p in ER.top_configs(st, K_CONFIGS) if p != c["hp"]][:K_CONFIGS - 1]
    P, y_oof = ER.lentes(fs, [c["hp"]] + outras, N_SEMENTES)
    tab, E = ER.classificar(P, y_oof)
    print(f"{P.shape[1]} lentes em {len(P)} clientes")
    print({k: round(comum.ks(y_oof.values, P[k].values), 4) for k in P})
    display(tab.tipo.value_counts().to_frame("clientes"))
    display(ER.resumo(tab, E).round(4))
    display(ER.concordancia(E).round(2))
    res = dict(perfil_fn=ER.perfil(tab, 1), perfil_fp=ER.perfil(tab, 0), risco=ER.mesmo_lado(tab, "risco"), fiel=ER.mesmo_lado(tab, "fiel"))
    for nome, titulo in (("perfil_fn", "Churners perdidos x churners pegos"), ("perfil_fp", "Fiéis acusados x fiéis reconhecidos"),
                         ("risco", "MESMO LADO, risco: fiéis acusados x churners pegos"), ("fiel", "MESMO LADO, fiel: churners perdidos x fiéis reconhecidos")):
        r = res[nome]
        print(titulo, f"| {int((r.q_bh < .05).sum())} segmentos com q < 0,05" if len(r) else "| sem dados")
        if len(r):
            display(r[r.q_bh < .05].assign(dif=lambda d: d.pct_nos_erros - d.pct_nos_acertos).sort_values("dif", key=abs, ascending=False).head(12).round(4))
    salvar_json(f"mlp_erros_{rotulo}.json", dict(lentes=list(P.columns), resumo=ER.resumo(tab, E).to_dict("records"), tipos=tab.tipo.value_counts().to_dict(),
                                                 indice=tab.index.tolist(), frac_erro=tab.frac_erro.tolist(), tipo=tab.tipo.tolist(),
                                                 **{k: v.to_dict("records") for k, v in res.items()}))


if continuar():
    analisar_erros(camp, "apos_busca")

11 lentes em 5282 clientes
{'mlp_melhor_s0': 0.5286, 'mlp_melhor_s1': 0.5273, 'mlp_melhor_s2': 0.5297, 'mlp_melhor_s3': 0.5284, 'mlp_melhor_s4': 0.5226, 'mlp_top2': 0.5295, 'mlp_top3': 0.5299, 'mlp_top4': 0.5269, 'mlp_top5': 0.5267, 'regressao_logistica': 0.5313, 'xgboost': 0.5204}


,clientes
tipo,
acerto consistente,3772
FP consistente,841
instavel,418
FN consistente,251


,classe,clientes,taxa_erro_media,todas_erram,esperado_se_independentes
0,churn (y=1),1402,0.2143,0.1320,0.0
1,nao churn (y=0),3880,0.2583,0.1905,0.0


,mlp_melhor_s0,mlp_melhor_s1,mlp_melhor_s2,mlp_melhor_s3,mlp_melhor_s4,mlp_top2,mlp_top3,mlp_top4,mlp_top5,regressao_logistica,xgboost
mlp_melhor_s0,1.00,0.84,0.92,0.93,0.88,0.84,0.84,0.84,0.87,0.91,0.68
mlp_melhor_s1,0.84,1.00,0.84,0.86,0.77,0.75,0.76,0.81,0.76,0.85,0.64
mlp_melhor_s2,0.92,0.84,1.00,0.93,0.88,0.82,0.82,0.82,0.85,0.90,0.69
mlp_melhor_s3,0.93,0.86,0.93,1.00,0.87,0.83,0.83,0.84,0.86,0.92,0.68
mlp_melhor_s4,0.88,0.77,0.88,0.87,1.00,0.86,0.85,0.81,0.90,0.87,0.70
mlp_top2,0.84,0.75,0.82,0.83,0.86,1.00,0.96,0.87,0.89,0.81,0.67
mlp_top3,0.84,0.76,0.82,0.83,0.85,0.96,1.00,0.90,0.88,0.81,0.66
mlp_top4,0.84,0.81,0.82,0.84,0.81,0.87,0.90,1.00,0.83,0.82,0.63
mlp_top5,0.87,0.76,0.85,0.86,0.90,0.89,0.88,0.83,1.00,0.84,0.69
regressao_logistica,0.91,0.85,0.90,0.92,0.87,0.81,0.81,0.82,0.84,1.00,0.68


Churners perdidos x churners pegos | 36 segmentos com q < 0,05


,segmento,n_erros,pct_nos_erros,pct_nos_acertos,p,q_bh,razao,dif
28,Contract_Month-to-month=1,118,0.4701,0.9913,0.0,0.0,0.4742,-0.5212
33,PaymentMethod_Electronic check=1,45,0.1793,0.6850,0.0,0.0,0.2617,-0.5057
26,InternetService_Fiber optic=1,89,0.3546,0.7967,0.0,0.0,0.4450,-0.4421
29,Contract_One year=1,96,0.3825,0.0087,0.0,0.0,44.1116,0.3738
43,PaymentMethod_Automatic=1,134,0.5339,0.1802,0.0,0.0,2.9634,0.3537
4,"tenure em (-0.001, 9.0]",58,0.2311,0.5761,0.0,0.0,0.4011,-0.3450
16,PaperlessBilling=1,132,0.5259,0.8256,0.0,0.0,0.6370,-0.2997
24,"TotalCharges em (3841.638, 8684.8]",91,0.3625,0.0684,0.0,0.0,5.3004,0.2941
19,"MonthlyCharges em (70.5, 89.988]",31,0.1235,0.4133,0.0,0.0,0.2988,-0.2898
7,"tenure em (55.75, 72.0]",76,0.3028,0.0145,0.0,0.0,20.9530,0.2883


... (3 outputs left out)

,segmento,n_erros,pct_nos_erros,pct_nos_acertos,p,q_bh,razao,dif
4,"tenure em (-0.001, 9.0]",340,0.4043,0.5761,0.0000,0.0000,0.7017,-0.1718
21,"TotalCharges em (-0.001, 391.062]",267,0.3175,0.4374,0.0000,0.0000,0.7259,-0.1199
26,InternetService_Fiber optic=1,586,0.6968,0.7967,0.0000,0.0000,0.8746,-0.0999
25,InternetService_DSL=1,253,0.3008,0.2013,0.0000,0.0000,1.4941,0.0995
33,PaymentMethod_Electronic check=1,497,0.5910,0.6850,0.0000,0.0002,0.8628,-0.0940
23,"TotalCharges em (1385.3, 3841.638]",251,0.2985,0.2091,0.0000,0.0001,1.4276,0.0894
6,"tenure em (29.0, 55.75]",173,0.2057,0.1224,0.0000,0.0000,1.6813,0.0834
43,PaymentMethod_Automatic=1,218,0.2592,0.1802,0.0000,0.0002,1.4389,0.0791
16,PaperlessBilling=1,629,0.7479,0.8256,0.0000,0.0002,0.9059,-0.0777
18,"MonthlyCharges em (35.45, 70.5]",247,0.2937,0.2274,0.0012,0.0040,1.2918,0.0663


MESMO LADO, fiel: churners perdidos x fiéis reconhecidos | 13 segmentos com q < 0,05


,segmento,n_erros,pct_nos_erros,pct_nos_acertos,p,q_bh,razao,dif
30,Contract_Two year=1,37,0.1474,0.4492,0.0000,0.0000,0.3282,-0.3017
28,Contract_Month-to-month=1,118,0.4701,0.2235,0.0000,0.0000,2.1036,0.2466
4,"tenure em (-0.001, 9.0]",58,0.2311,0.1046,0.0000,0.0000,2.2090,0.1265
26,InternetService_Fiber optic=1,89,0.3546,0.2286,0.0000,0.0002,1.5511,0.1260
21,"TotalCharges em (-0.001, 391.062]",68,0.2709,0.1551,0.0000,0.0001,1.7469,0.1158
7,"tenure em (55.75, 72.0]",76,0.3028,0.4151,0.0005,0.0029,0.7294,-0.1124
27,InternetService_No=1,68,0.2709,0.3782,0.0008,0.0042,0.7163,-0.1073
20,"MonthlyCharges em (89.988, 118.75]",77,0.3068,0.1997,0.0001,0.0009,1.5361,0.1071
22,"TotalCharges em (391.062, 1385.3]",39,0.1554,0.2465,0.0009,0.0042,0.6303,-0.0911
15,StreamingMovies=1,110,0.4382,0.3508,0.0072,0.0317,1.2494,0.0875


GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



### 📝 O que os erros mostram

- **Os erros são compartilhados entre famílias ou são do MLP?** _…_
- **O que separa clientes do mesmo lado com rótulos opostos?** _…_
- **Que hipótese isso gera para o próximo bloco?** _…_

# Insights em sequência

Cada bloco testa uma intervenção sobre o campeão do bloco anterior. **Para acrescentar um insight:** coloque a variável em `BL.CANDIDATAS`, copie um bloco abaixo, dê a ele o próximo número e registre de onde veio a ideia.

**Estes blocos são a sequência do MLP.** As dobras oficiais não mudam, e os outros modelos não são afetados. Uma mudança só vale para todos se o grupo decidir.

As variações abaixo são o ponto de partida, vindas da sondagem inicial. Muitos blocos em sequência aumentam a chance de aceitar algo por acaso, e por isso a régua é exigente e o teste fica para o fim.

## b02_oversampling

**Pergunta:** Tirar o oversampling melhora o KS?

In [10]:
if continuar():
    camp, t = BL.rodar("b02_oversampling", "Tirar o oversampling melhora o KS?", {"sem oversampling": dict(dados=dict(over=False))}, camp)
    display(t.round(4))

b02_oversampling: Tirar o oversampling melhora o KS?
regua de ruido = 0.0028 | campeao mantido | 154s
GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



,variante,ks,delta,dp_delta,ganhou,decisao
0,(campeao atual),0.5400,0.0000,0.0000,,base
1,sem oversampling,0.5413,0.0014,0.0033,4/5,rejeitada


## b03_variaveis_criadas

**Pergunta:** As variáveis criadas no pré-processamento ajudam, agora com os dados na escala certa?

In [11]:
if continuar():
    camp, t = BL.rodar("b03_variaveis_criadas", "As variáveis criadas no pré-processamento ajudam, agora com os dados na escala certa?", {"sem total_addons e PaymentMethod_Automatic": dict(dados=dict(criadas=False))}, camp)
    display(t.round(4))

b03_variaveis_criadas: As variáveis criadas no pré-processamento ajudam, agora com os dados na escala certa?
regua de ruido = 0.0028 | campeao mantido | 135s
GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



,variante,ks,delta,dp_delta,ganhou,decisao
0,(campeao atual),0.5400,0.0000,0.0000,,base
1,sem total_addons e PaymentMethod_Automatic,0.5383,-0.0016,0.0032,1/5,rejeitada


## b04_cliente_novo

**Pergunta:** Marcar clientes novos (até 6 meses) ajuda?

In [12]:
if continuar():
    camp, t = BL.rodar("b04_cliente_novo", "Marcar clientes novos (até 6 meses) ajuda?", {"cliente_novo": dict(extras=["cliente_novo"])}, camp)
    display(t.round(4))

b04_cliente_novo: Marcar clientes novos (até 6 meses) ajuda?
regua de ruido = 0.0028 | novo campeao | 129s
GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



,variante,ks,delta,dp_delta,ganhou,decisao
0,(campeao atual),0.5400,0.0000,0.0000,,base
1,cliente_novo,0.5491,0.0092,0.0036,5/5,ACEITA


## b05_delta_preco

**Pergunta:** A diferença entre o preço atual e a média já paga ajuda?

In [13]:
if continuar():
    camp, t = BL.rodar("b05_delta_preco", "A diferença entre o preço atual e a média já paga ajuda?", {"delta_preco": dict(extras=["delta_preco"])}, camp)
    display(t.round(4))

b05_delta_preco: A diferença entre o preço atual e a média já paga ajuda?
regua de ruido = 0.0009 | novo campeao | 126s
GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



,variante,ks,delta,dp_delta,ganhou,decisao
0,(campeao atual),0.5491,0.0000,0.0000,,base
1,delta_preco,0.5506,0.0014,0.0022,4/5,ACEITA


## b06_mtm_fibra

**Pergunta:** A interação entre contrato mensal e fibra ajuda?

In [14]:
if continuar():
    camp, t = BL.rodar("b06_mtm_fibra", "A interação entre contrato mensal e fibra ajuda?", {"mtm_fibra": dict(extras=["mtm_fibra"])}, camp)
    display(t.round(4))

b06_mtm_fibra: A interação entre contrato mensal e fibra ajuda?
regua de ruido = 0.0019 | campeao mantido | 127s
GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



,variante,ks,delta,dp_delta,ganhou,decisao
0,(campeao atual),0.5506,0.0000,0.0000,,base
1,mtm_fibra,0.5510,0.0004,0.0014,4/5,rejeitada


## b07_preco_por_servico

**Pergunta:** O preço por serviço assinado ajuda?

In [15]:
if continuar():
    camp, t = BL.rodar("b07_preco_por_servico", "O preço por serviço assinado ajuda?", {"preco_por_servico": dict(extras=["preco_por_servico"])}, camp)
    display(t.round(4))

b07_preco_por_servico: O preço por serviço assinado ajuda?
regua de ruido = 0.0019 | campeao mantido | 130s
GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



,variante,ks,delta,dp_delta,ganhou,decisao
0,(campeao atual),0.5506,0.0000,0.0000,,base
1,preco_por_servico,0.5498,-0.0008,0.0016,2/5,rejeitada


## Evolução do campeão

O que cada bloco testou, o que foi aceito e quanto mudou. É esta tabela que responde o que melhora o modelo e o que não melhora.

In [16]:
evo = BL.historico()
display(evo.round(4))
print("Campeão atual:", json.dumps(camp, ensure_ascii=False))
salvar_json("mlp_historico_blocos.json", evo.to_dict("records"))

,bloco,pergunta,ks_base,regua,aceita,delta
0,b01_busca,O melhor trial supera o MLP padrão em divisões novas?,0.5400,0.0028,-,0.0000
1,b02_oversampling,Tirar o oversampling melhora o KS?,0.5400,0.0028,-,0.0000
2,b03_variaveis_criadas,"As variáveis criadas no pré-processamento ajudam, agora ...",0.5400,0.0028,-,0.0000
3,b04_cliente_novo,Marcar clientes novos (até 6 meses) ajuda?,0.5400,0.0028,cliente_novo,0.0092
4,b05_delta_preco,A diferença entre o preço atual e a média já paga ajuda?,0.5491,0.0009,delta_preco,0.0014
5,b06_mtm_fibra,A interação entre contrato mensal e fibra ajuda?,0.5506,0.0019,-,0.0000
6,b07_preco_por_servico,O preço por serviço assinado ajuda?,0.5506,0.0019,-,0.0000


Campeão atual: {"hp": {"camadas": 1, "unidades": 10, "ativ": "relu", "otim": "adam", "lr": 0.001, "dropout": 0.0, "l2": 1e-06, "perda": "ce", "paciencia": 10}, "dados": {}, "extras": ["cliente_novo", "delta_preco"]}
GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



In [17]:
# Os erros mudaram depois dos insights?
if continuar():
    analisar_erros(camp, "apos_insights")

11 lentes em 5282 clientes
{'mlp_melhor_s0': 0.5379, 'mlp_melhor_s1': 0.5342, 'mlp_melhor_s2': 0.5369, 'mlp_melhor_s3': 0.5362, 'mlp_melhor_s4': 0.5325, 'mlp_top2': 0.5383, 'mlp_top3': 0.5355, 'mlp_top4': 0.536, 'mlp_top5': 0.5392, 'regressao_logistica': 0.5332, 'xgboost': 0.5146}


,clientes
tipo,
acerto consistente,3804
FP consistente,861
instavel,373
FN consistente,244


,classe,clientes,taxa_erro_media,todas_erram,esperado_se_independentes
0,churn (y=1),1402,0.2091,0.1412,0.0
1,nao churn (y=0),3880,0.2569,0.1925,0.0


,mlp_melhor_s0,mlp_melhor_s1,mlp_melhor_s2,mlp_melhor_s3,mlp_melhor_s4,mlp_top2,mlp_top3,mlp_top4,mlp_top5,regressao_logistica,xgboost
mlp_melhor_s0,1.00,0.91,0.88,0.91,0.83,0.87,0.87,0.84,0.91,0.83,0.70
mlp_melhor_s1,0.91,1.00,0.85,0.89,0.81,0.84,0.86,0.81,0.89,0.81,0.70
mlp_melhor_s2,0.88,0.85,1.00,0.88,0.89,0.89,0.88,0.88,0.88,0.87,0.68
mlp_melhor_s3,0.91,0.89,0.88,1.00,0.86,0.89,0.89,0.86,0.91,0.86,0.69
mlp_melhor_s4,0.83,0.81,0.89,0.86,1.00,0.88,0.88,0.89,0.86,0.87,0.67
mlp_top2,0.87,0.84,0.89,0.89,0.88,1.00,0.95,0.95,0.89,0.84,0.67
mlp_top3,0.87,0.86,0.88,0.89,0.88,0.95,1.00,0.93,0.90,0.83,0.68
mlp_top4,0.84,0.81,0.88,0.86,0.89,0.95,0.93,1.00,0.86,0.83,0.67
mlp_top5,0.91,0.89,0.88,0.91,0.86,0.89,0.90,0.86,1.00,0.82,0.69
regressao_logistica,0.83,0.81,0.87,0.86,0.87,0.84,0.83,0.83,0.82,1.00,0.67


Churners perdidos x churners pegos | 35 segmentos com q < 0,05


,segmento,n_erros,pct_nos_erros,pct_nos_acertos,p,q_bh,razao,dif
28,Contract_Month-to-month=1,115,0.4713,0.9887,0.0,0.0,0.4767,-0.5174
33,PaymentMethod_Electronic check=1,48,0.1967,0.6814,0.0,0.0,0.2887,-0.4847
26,InternetService_Fiber optic=1,84,0.3443,0.7898,0.0,0.0,0.4359,-0.4456
4,"tenure em (-0.001, 9.0]",49,0.2008,0.5815,0.0,0.0,0.3453,-0.3807
29,Contract_One year=1,92,0.3770,0.0113,0.0,0.0,33.3374,0.3657
43,PaymentMethod_Automatic=1,132,0.5410,0.1791,0.0,0.0,3.0210,0.3619
16,PaperlessBilling=1,127,0.5205,0.8200,0.0,0.0,0.6348,-0.2995
24,"TotalCharges em (3841.638, 8684.8]",86,0.3525,0.0782,0.0,0.0,4.5055,0.2742
7,"tenure em (55.75, 72.0]",71,0.2910,0.0198,0.0,0.0,14.7016,0.2712
19,"MonthlyCharges em (70.5, 89.988]",33,0.1352,0.4043,0.0,0.0,0.3345,-0.2691


... (3 outputs left out)

,segmento,n_erros,pct_nos_erros,pct_nos_acertos,p,q_bh,razao,dif
4,"tenure em (-0.001, 9.0]",360,0.4181,0.5815,0.0000,0.0000,0.7190,-0.1634
21,"TotalCharges em (-0.001, 391.062]",288,0.3345,0.4486,0.0000,0.0000,0.7456,-0.1141
33,PaymentMethod_Electronic check=1,507,0.5889,0.6814,0.0000,0.0001,0.8641,-0.0926
23,"TotalCharges em (1385.3, 3841.638]",253,0.2938,0.2036,0.0000,0.0000,1.4434,0.0903
6,"tenure em (29.0, 55.75]",182,0.2114,0.1254,0.0000,0.0000,1.6863,0.0860
26,InternetService_Fiber optic=1,610,0.7085,0.7898,0.0000,0.0002,0.8970,-0.0813
25,InternetService_DSL=1,241,0.2799,0.1998,0.0000,0.0002,1.4009,0.0801
43,PaymentMethod_Automatic=1,221,0.2567,0.1791,0.0000,0.0002,1.4333,0.0776
16,PaperlessBilling=1,640,0.7433,0.8200,0.0001,0.0002,0.9065,-0.0767
10,OnlineSecurity=1,145,0.1684,0.1008,0.0000,0.0001,1.6699,0.0676


MESMO LADO, fiel: churners perdidos x fiéis reconhecidos | 14 segmentos com q < 0,05


,segmento,n_erros,pct_nos_erros,pct_nos_acertos,p,q_bh,razao,dif
30,Contract_Two year=1,37,0.1516,0.4459,0.0000,0.0000,0.3401,-0.2942
28,Contract_Month-to-month=1,115,0.4713,0.2275,0.0000,0.0000,2.0718,0.2438
27,InternetService_No=1,58,0.2377,0.3682,0.0000,0.0004,0.6456,-0.1305
26,InternetService_Fiber optic=1,84,0.3443,0.2249,0.0001,0.0005,1.5305,0.1193
7,"tenure em (55.75, 72.0]",71,0.2910,0.4087,0.0003,0.0021,0.7120,-0.1177
4,"tenure em (-0.001, 9.0]",49,0.2008,0.0900,0.0000,0.0000,2.2302,0.1108
15,StreamingMovies=1,108,0.4426,0.3500,0.0043,0.0208,1.2647,0.0926
20,"MonthlyCharges em (89.988, 118.75]",70,0.2869,0.1947,0.0011,0.0061,1.4736,0.0922
21,"TotalCharges em (-0.001, 391.062]",57,0.2336,0.1422,0.0003,0.0021,1.6430,0.0914
3,Dependents=1,79,0.3238,0.4131,0.0065,0.0285,0.7839,-0.0893


GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



### 📝 Análise da sequência

- **O que foi aceito e o que foi rejeitado?** _…_
- **Os erros consistentes mudaram?** _…_
- **Algum achado vale levar para o grupo?** _…_

# Registro evolutivo do pré-processamento

O que cada etapa muda, de forma cumulativa, no MLP e no XGBoost. Cada `delta` é contra a etapa anterior, pareado por divisão do treino. Imputação e codificação ficam sempre, porque sem elas os modelos não rodam. O MLP usa a melhor configuração do estudo, e o XGBoost, uma configuração fixa.

In [18]:
if continuar():
    r_evo = EV.registro(st.best_params, REPETICOES)
    display(r_evo.round(4))
    salvar_json("mlp_evolucao.json", r_evo.to_dict("records"))

,modelo,variante,ks,dp_ks,delta,dp_delta,ganhou
0,mlp,codificacao,0.4398,0.0454,NaN,NaN,
1,mlp,+ variaveis criadas,0.4602,0.0347,0.0203,0.0702,3/5
2,mlp,+ StandardScaler,0.5440,0.0024,0.0838,0.0353,5/5
3,mlp,+ oversampling (oficial),0.5408,0.0041,-0.0032,0.0030,0/5
4,xgboost,codificacao,0.5381,0.0036,NaN,NaN,
5,xgboost,+ variaveis criadas,0.5364,0.0025,-0.0017,0.0026,1/5
6,xgboost,+ StandardScaler,0.5364,0.0025,0.0000,0.0000,0/5
7,xgboost,+ oversampling (oficial),0.5321,0.0059,-0.0044,0.0045,0/5


GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



### 📝 Análise do registro evolutivo

- **Qual etapa mais mudou o MLP? E o XGBoost?** _…_

# Avaliação final no teste

Só com a sequência congelada, e uma única vez. Saem duas linhas:

- **`mlp.json`:** a melhor configuração do Optuna com os dados oficiais. É a linha **comparável com os outros modelos**.
- **`mlp_campeao.json`:** o campeão da sequência de blocos, com o que foi aceito. Mostra quanto os insights renderam fora da validação.

Nas duas, o modelo é treinado no treino inteiro, em 5 sementes, pela mediana das melhores épocas nas dobras, e o limiar sai da validação.

In [19]:
if REVELAR_TESTE and continuar():
    A.main(N_SEMENTES)
    r = json.load(open(os.path.join(PASTA, "mlp.json")))
    fin = BL.final_campeao(camp, N_SEMENTES)
    salvar_json("mlp_campeao.json", fin)
    print(f"Oficial (Optuna): KS = {r['teste']['ks']:.4f} | IC 95% {np.round(r['teste']['ks_ic95'], 4)} | {r['melhor_config']}")
    print(f"Campeão dos blocos: KS = {fin['teste']['ks']:.4f} | IC 95% {np.round(fin['teste']['ks_ic95'], 4)} | {camp}")
    display(pd.DataFrame(r["ce_vs_mse"]["teste"]).T[["ks", "auroc", "ce", "mse", "ce_corr", "f1", "precision", "recall"]].round(4))
else:
    print("Teste não revelado (REVELAR_TESTE = False).")

Teste não revelado (REVELAR_TESTE = False).


In [20]:
salvar()
if wandb_run is not None:
    wandb_run.summary.update(dict(melhor_ks_validacao=st.best_value, **{f"melhor/{k}": v for k, v in st.best_params.items()}))
    wandb_run.finish()
print("Resultados em", PASTA)
print(sorted(os.listdir(PASTA)))

wandb: updating run metadata


GitHub: falha ao enviar os resultados: remote: Permission to diegoflyra/churn_project.git denied to diegoflyra.
fatal: unable to access 'https://github.com/diegoflyra/churn_project.git/': The requested URL returned error: 403



wandb: uploading summary, console lines 65-71
wandb: 
wandb: Run history:
wandb:   hp/camadas ▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:   hp/dropout ▅▃▃▃▁█▁▃▂▁▁▂▂▃▂▂▂▃▁▄▄▂▃▃▆▃▆▃▄▃▅▂▄▃▄▃▃▂▄▃
wandb:        hp/l2 ▇▁▁▃▆▂▃▃▁▇▄▂▂▄▆█▃▂▅▇▃▃▅▄▅▂▂▂▁▂▃▁▁▁▂▃▁▂▂▂
wandb:        hp/lr ▃█▂▆▁▂▂▁▁▁▁▃▁▂▁▁▁▁▁▃▁▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: hp/paciencia ▁▅▁▁▂▂▂▁▂▂▂▁▂█▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
wandb:  hp/unidades ▁▁▁█▆▁▁▁▂▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:           ks ▆▁▆▆▆████▆▇█▇▆▆█▆████▆█▇▇▆▆▇███▇▆▆▆▆█▆▆█
wandb:   ks_campeao ▁▁▁▇███
wandb:    melhor_ks ▁▁▁▁▄▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▇▇▇▇████████████
wandb:        regua ████▁▅▅
wandb:           +1 ...
wandb: 
wandb: Run summary:
wandb:        bloco b07_preco_por_servic...
wandb:       estado PRUNED
wandb:      hp/ativ relu
wandb:   hp/camadas 1
wandb:   hp/dropout 0.19287
wandb:        hp/l2 0.00127
wandb:        hp/lr 0.00022
wandb:      hp/otim adam
wandb: hp/paciencia 10
wandb:     hp/perda ce
wandb:          +16 ...
wandb: 
wandb: 🚀 View run mlp-202

Resultados em /kaggle/working/resultados_mlp-20261103-0337
['blocos', 'mlp.db', 'mlp_erros_apos_busca.json', 'mlp_erros_apos_insights.json', 'mlp_evolucao.json', 'mlp_historico_blocos.json']
